In [18]:
import hoda
import tensorly as tl

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [19]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation



paradigm = P300(resample=64)
datasets = [
       #BI2012(),
       #BI2014a(),
       #BI2014b(),
       #BI2015a(),
       #BI2015b(),
       #BNCI2014008(),
       #BNCI2014009(),
       #BNCI2015_003(),
       #Cattan2019_VR(),
       EPFLP300(),
       #Huebner2017(),
       #Huebner2018(),
       #Lee2019_ERP(),
       #Sosulski2019()   
   ]



In [20]:
from sklearn.pipeline import Pipeline
from mne.decoding import Scaler
from hoda.hoda import  HODA, GreedyBTTDA
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, GridSearchCV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hoda_params = dict(
    max_iter=128,
    tol=1e-12,
    init ='random',
    shrinkage='lw',
    toeplitz=None,
    obj='rt',
    ortho=False,
    solver='lanczos',
    taper=False,
    extra_train_info=False,
    verbose=False, 
)

bttda = GreedyBTTDA(
        max_blocks=16,
        hoda_params=hoda_params,
        verbose=True,
        extra_train_info=False,
        gs_params=dict(n_jobs=-1, cv=cv),
        truncate=True,
)



scaler = Scaler(scalings='mean', with_mean=True)

In [22]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import dask


%env PYTHONWARNINGS=ignore

test_size = 0.2

df = []
for dataset in datasets:
    X_sub, y_sub, meta = paradigm.get_data(dataset, 
        #subjects=[1]
    )
    idc = meta.reset_index()\
        .groupby(['subject', 'session'])\
        .index.aggregate(list)
    
    for session, session_idc in idc.items():
        print(dataset.code + ' ' + str(session))
        session_idc = np.array(session_idc) 

        
        X =X_sub[session_idc]
        y = y_sub[session_idc]
        X = scaler.fit_transform(X)
        
        X = tl.tensor(X)

        X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=test_size, shuffle=True, random_state=42)

        bttda.fit(X_train,y_train, X_test=X_test, y_test=y_test)
        df_session = bttda.train_info_
        df_session['dataset'] = dataset.code
        df_session['subject'] = session[0]
        df_session['session'] = session[1]
        df.append(df_session)
df = pd.concat([pd.DataFrame(d) for d in df], axis=0)

env: PYTHONWARNINGS=ignore


/usr/local/lib/python3.10/dist-packages/moabb/datasets/preprocessing.py:85: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annotations)
/usr/local/lib/python3.10/dist-packages/moabb/datasets/preprocessing.py:85: RuntimeWarning: Limited 3 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annotations)
/usr/local/lib/python3.10/dist-packages/moabb/datasets/preprocessing.py:85: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annotations)
/usr/local/lib/python3.10/dist-packages/moabb/datasets/preprocessing.py:85: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annotations)
/usr/local/lib/python3.10/dist-packages/moabb/datasets/preprocessing.py:85: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annotations)
/usr/local/lib/python3.10

EPFLP300 (1, '0')
Fitting block 1/16...


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True 

Fitting block 2/16...
Fitting block 3/16...


...

Fitting block 5/16...


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py:547: FitFailedWarning: 
19 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
19 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/pipeline.py", line 471, in fit
    Xt = self._fit(X, y, routed_params)
  File "/usr/local/lib/python3.10/dist-packages/skl

EPFLP300 (1, '3')
Fitting block 1/16...


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py:547: FitFailedWarning: 
7 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/pipeline.py", line 471, in fit
    Xt = self._fit(X, y, routed_params)
  File "/usr/local/lib/python3.10/dist-packages/sklea

CUBLASError: CUBLAS_STATUS_EXECUTION_FAILED

In [ ]:
pd.set_option("display.max_rows", None)
df

In [ ]:
import math
df['n_features'] = df['rank'].apply(math.prod)
df['n_features'] = df.groupby(['dataset', 'subject', 'session'])['n_features'].cumsum()
df

In [ ]:
df_diff = df.reset_index()
idx = ['dataset', 'subject', 'session']
df_diff = df_diff.set_index(idx)

df_test_score_first = df_diff.groupby(['dataset', 'subject', 'session']).test_score.aggregate('first')
df_diff['test_score_diff_hoda'] = df_diff['test_score'] - df_test_score_first
df_val_score_first = df_diff.groupby(['dataset', 'subject', 'session']).val_score.aggregate('first')
df_diff['val_score_diff_hoda'] = df_diff['val_score'] - df_val_score_first

df_diff = df_diff.reset_index()
df_diff

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


sns.relplot(
    data=df_diff,
    col='dataset',
    #col_wrap=4,
    x='block',
    y='test_score_diff_hoda',
    hue='subject',
    kind="line",
    palette="tab10",
    #ci=None,
    #units='fold',
    #estimator=None,
)
plt.axhline(0, color='black')
plt.show()



In [ ]:
sns.relplot(
    data=df_diff,
    col='dataset',
    #col_wrap=4,
    x='block',
    y='val_score_diff_hoda',
    hue='subject',
    kind="line",
    palette="tab10",
    #ci=None,
    #units='fold',
    #estimator=None,
)
plt.show()

In [ ]:
sns.relplot(
    data=df_diff,
    col='dataset',
    #col_wrap=4,
    x='block',
    y='test_score',
    hue='session',
    kind="line",
    palette="tab10",
    #ci=None,
    #units='fold',
    #estimator=None,
)
plt.show()

To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.
To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.
To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.
To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.
To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.
To use the get_shape_from_baseconcar, InputShapeSetterE